<a href="https://colab.research.google.com/github/Feellived/molecular-reliability-signals/blob/yoonsoo/A4_hierarchical_followup(fixed).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A-4 계층적 분산 분해 후속 (수정판)

원본 노트북이 채점 단계에서 멈추던 원인을 고쳤다.

## 무엇이 문제였나

**`Path.rglob` 이 구글 드라이브 마운트에서 동작하지 않는다.** 바로가기를 통과하지 못해
빈 결과를 돌려준다. 원본은 탐색을 전부 `rglob` 으로 했다.

```
노트북 셀           rglob 3회   -> 채점기·체크포인트·splits 후보 0개
score_nested_a4.py  rglob 3회   -> 같은 실패가 스크립트 안에서 반복
compare_approx_formal.py rglob 2회
```

`score_variants_chemberta.py` 는 `Main/scripts_role4/` 에 분명히 있는데도 "없음" 으로 나왔다.

## 무엇을 고쳤나

| | |
|---|---|
| 탐색 방식 | `rglob` → `iterdir` 재귀(`walk_find`). 바로가기를 통과한다 |
| splits 위치 | `--splits-root` 인자 추가. 지정하면 거기만 본다 |
| 스크립트 보관 | base64 압축 덩어리 → **평문**. 내용을 읽고 고칠 수 있다 |

채점 로직(토큰 길이·확률·회귀 역변환)은 **건드리지 않았다.** 담당4의
`score_variants_chemberta.py` 원문을 그대로 복사해 호출하는 구조도 그대로다.

## splits 경로

```
Conference_2026/Juhyeong/data/processed/pipeline_yoonsoo/<물성>/splits.csv
```

In [1]:
# ============================================================================
# [셀 1] 설치 + 마운트 + 경로
# ============================================================================
%pip install -q transformers scipy matplotlib

In [2]:
import warnings; warnings.filterwarnings("ignore")
import json, re, subprocess, sys, shutil
from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/Conference_2026")
PERSONAL_ROOT = PROJECT_ROOT / "Yoonsoo"
VARIANTS_DIR = PERSONAL_ROOT / "variants_a4_full"

# 물성별 splits 가 있는 곳
SPLITS_ROOT = PROJECT_ROOT / "Juhyeong" / "data" / "processed" / "pipeline_yoonsoo"

# 체크포인트. 시드가 42/43/44 세 벌인데 기존 채점과 같은 것을 써야 한다.
# 다른 시드로 채점하면 5단계 비교에서 '정의 차이'와 '모델 차이'가 섞인다.
# 대조 결과 seed_42 가 기존 예측을 평균절대차 2e-8 로 재현했다(43: 0.0155, 44: 0.0131).
CHECKPOINT_ROOT = PROJECT_ROOT / "Jiye" / "checkpoints" / "chemberta_seed_42" / "checkpoints"

# 담당4가 원래 채점한 결과. 시드 검증과 5단계 비교에 쓴다.
OLD_SCORES = PROJECT_ROOT / "Juhyeong" / "data" / "processed" / "scores_role4" / "chemberta"

SCORES_DIR  = PERSONAL_ROOT / "scores_a4_full"
DECOMP_DIR  = PERSONAL_ROOT / "a4_decomposition_full"
COMPARE_DIR = PERSONAL_ROOT / "a4_comparison_full"

WORK_DIR = Path("/content/a4_followup")      # 로컬. 드라이브에 쓰면 느리다
WORK_DIR.mkdir(parents=True, exist_ok=True)

for label, p in [("A-4 입력", VARIANTS_DIR), ("splits", SPLITS_ROOT),
                 ("체크포인트", CHECKPOINT_ROOT), ("기존 채점", OLD_SCORES),
                 ("개인 공간", PERSONAL_ROOT), ("팀 루트", PROJECT_ROOT)]:
    print(f"{label:10s} {'있음' if p.exists() else '없음'}  {p}")

assert VARIANTS_DIR.is_dir(), f"A-4 입력 없음: {VARIANTS_DIR}"
assert SPLITS_ROOT.is_dir(), f"splits 루트 없음: {SPLITS_ROOT}"
assert CHECKPOINT_ROOT.is_dir(), f"체크포인트 없음: {CHECKPOINT_ROOT}"

Mounted at /content/drive
A-4 입력     있음  /content/drive/MyDrive/Conference_2026/Yoonsoo/variants_a4_full
splits     있음  /content/drive/MyDrive/Conference_2026/Juhyeong/data/processed/pipeline_yoonsoo
체크포인트      있음  /content/drive/MyDrive/Conference_2026/Jiye/checkpoints/chemberta_seed_42/checkpoints
기존 채점      있음  /content/drive/MyDrive/Conference_2026/Juhyeong/data/processed/scores_role4/chemberta
개인 공간      있음  /content/drive/MyDrive/Conference_2026/Yoonsoo
팀 루트       있음  /content/drive/MyDrive/Conference_2026


## 1단계 — 탐색

`walk_find` 로 채점기·체크포인트·splits 를 찾는다. 셋 다 찾아야 진행한다.

In [5]:
# ============================================================================
# [셀 3] 실제 경로 확인
# ----------------------------------------------------------------------------
# rglob 대신 iterdir 재귀. 드라이브 바로가기를 통과한다.
# ============================================================================
_SKIP = {"__pycache__", ".ipynb_checkpoints", ".git"}

def walk_find(root, name=None, suffix=None, max_depth=8):
    found = []
    def rec(d, depth):
        if depth > max_depth:
            return
        try:
            entries = list(d.iterdir())
        except Exception:
            return
        for e in entries:
            try:
                is_dir = e.is_dir()
            except Exception:
                continue
            if is_dir:
                if e.name not in _SKIP:
                    rec(e, depth + 1)
            elif (name is not None and e.name == name) or \
                 (suffix is not None and e.suffix.lower() == suffix) or \
                 (name is None and suffix is None):
                found.append(e)
    rec(Path(root), 0)
    return found


datasets = sorted(p.parent.name for p in VARIANTS_DIR.glob("*/variants_nested.csv"))
print("대상 물성:", datasets, "\n")

# --- 채점기 ---
scorers = walk_find(PROJECT_ROOT, name="score_variants_chemberta.py")
print("채점기 후보:")
for p in scorers:
    print("   ", p.relative_to(PROJECT_ROOT))
SCORER_PATH = scorers[0] if len(scorers) == 1 else None
if len(scorers) > 1:
    import hashlib
    h = {hashlib.sha256(p.read_bytes()).hexdigest() for p in scorers}
    SCORER_PATH = sorted(scorers, key=lambda p: len(p.parts))[0] if len(h) == 1 else None
    print("   내용 동일" if len(h) == 1 else "   !! 내용이 다르다. 직접 골라야 한다")

# --- 체크포인트 (시드 확정됨. 구조만 확인한다) ---
miss = [f"{ds}/{v}" for ds in datasets for v in ("regular", "augmented")
        if not (CHECKPOINT_ROOT / ds / v / "complete.json").is_file()]
print(f"\n체크포인트: {CHECKPOINT_ROOT.name} | 누락 {len(miss)}개")
if miss:
    print("   ", miss[:6])

# --- splits ---
print("\nsplits:")
for ds in datasets:
    f = SPLITS_ROOT / ds / "splits.csv"
    print(f"   {'O' if f.exists() else 'X'} {ds}")

print("\n=== 확정 ===")
print("SCORER_PATH     :", SCORER_PATH or "!! 미확정")
print("CHECKPOINT_ROOT :", CHECKPOINT_ROOT)

대상 물성: ['ames', 'bbb_martins', 'bioavailability_ma', 'caco2_wang', 'clearance_hepatocyte_az', 'clearance_microsome_az', 'cyp2c9_substrate_carbonmangels', 'cyp2c9_veith', 'cyp2d6_substrate_carbonmangels', 'cyp2d6_veith', 'cyp3a4_substrate_carbonmangels', 'cyp3a4_veith', 'dili', 'half_life_obach', 'herg', 'hia_hou', 'ld50_zhu', 'lipophilicity_astrazeneca', 'pgp_broccatelli', 'ppbr_az', 'solubility_aqsoldb', 'vdss_lombardo'] 

채점기 후보:
    Juhyeong/scripts/score_variants_chemberta.py
    Main/scripts_role4/score_variants_chemberta.py
   내용 동일

체크포인트: checkpoints | 누락 0개

splits:
   O ames
   O bbb_martins
   O bioavailability_ma
   O caco2_wang
   O clearance_hepatocyte_az
   O clearance_microsome_az
   O cyp2c9_substrate_carbonmangels
   O cyp2c9_veith
   O cyp2d6_substrate_carbonmangels
   O cyp2d6_veith
   O cyp3a4_substrate_carbonmangels
   O cyp3a4_veith
   O dili
   O half_life_obach
   O herg
   O hia_hou
   O ld50_zhu
   O lipophilicity_astrazeneca
   O pgp_broccatelli
   O ppbr_az

## 시드 확인

체크포인트가 `chemberta_seed_42/43/44` 세 벌이고 **셋 다 22물성 × regular·augmented 로 완전하다.**
아무거나 쓰면 안 된다 — 다른 시드로 채점하면 5단계 비교에서 *정의 차이* 와 *모델 차이* 가 섞인다.

담당4가 원래 채점한 결과(`scores_role4/chemberta`)를 각 시드로 재현해 본 결과다.

| 시드 | 평균절대차 |
|---|---|
| **42** | **0.00000002** <- 재현됨 |
| 43 | 0.01547592 |
| 44 | 0.01309913 |

그래서 `seed_42` 로 고정했다. 아래 셀이 실행 시점에 다시 확인한다.

In [ ]:
# ============================================================================
# [셀 4] 시드 검증 - seed_42 가 기존 채점을 재현하는가
# ----------------------------------------------------------------------------
# 체크포인트가 바뀌면 에러 없이 다른 값이 나온다. 200분자로 빠르게 확인한다.
# ============================================================================
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DS_CHECK = "herg"        # 분류 물성이라 회귀 역변환이 끼지 않는다

old = pd.read_csv(OLD_SCORES / DS_CHECK / "origin_predictions_chemberta.csv")
sp  = pd.read_csv(SPLITS_ROOT / DS_CHECK / "splits.csv", low_memory=False)
chk = old.merge(sp[["row_uid", "parent_smiles"]], on="row_uid").head(200)

with torch.inference_mode():
    ck  = CHECKPOINT_ROOT / DS_CHECK / "regular"
    tok = AutoTokenizer.from_pretrained(str(ck))
    mod = AutoModelForSequenceClassification.from_pretrained(str(ck)).eval()
    mod = mod.cuda() if torch.cuda.is_available() else mod
    enc = tok(list(chk.parent_smiles), padding=True, truncation=True,
              max_length=128, return_tensors="pt")
    enc = {k: v.to(mod.device) for k, v in enc.items()}
    pred = torch.softmax(mod(**enc).logits, -1)[:, 1].cpu().numpy()

diff = abs(pred - chk.pred_chemberta_regular).mean()
print(f"{DS_CHECK} 200분자 평균절대차: {diff:.8f}")
assert diff < 1e-4, "기존 채점과 다른 체크포인트다. 시드를 다시 확인한다"
print("확인: 기존 채점과 같은 모델")

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

herg 200분자 평균절대차: 0.00000002
확인: 기존 채점과 같은 모델


## 2단계 - 스크립트 작성

원본의 base64 덩어리를 평문으로 풀고 `rglob` 만 고쳤다. 내용을 직접 읽을 수 있다.

In [ ]:
%%writefile /content/a4_followup/score_nested_a4.py
#!/usr/bin/env python
"""Prepare A-4 nested inputs and invoke the existing ChemBERTa scorer unchanged."""

from __future__ import annotations

import argparse
import hashlib
import json
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd



# ---------------------------------------------------------------------------
# 구글 드라이브 마운트에서 Path.rglob 는 바로가기를 통과하지 못해 빈 결과를 준다.
# iterdir 재귀로 같은 일을 한다. 이 파일의 모든 탐색은 이 함수를 쓴다.
# ---------------------------------------------------------------------------
_SKIP_DIRS = {"__pycache__", ".ipynb_checkpoints", ".git"}


def walk_find(root, name=None, suffix=None, max_depth=8):
    """root 아래 파일을 재귀로 찾는다. name 정확 일치 또는 suffix 로 거른다."""
    found = []

    def rec(d, depth):
        if depth > max_depth:
            return
        try:
            entries = list(d.iterdir())
        except Exception:
            return
        for e in entries:
            try:
                is_dir = e.is_dir()
            except Exception:
                continue
            if is_dir:
                if e.name not in _SKIP_DIRS:
                    rec(e, depth + 1)
            elif name is not None:
                if e.name == name:
                    found.append(e)
            elif suffix is not None:
                if e.suffix.lower() == suffix:
                    found.append(e)
            else:
                found.append(e)

    rec(Path(root), 0)
    return found

KEEP_COLUMNS = [
    "variant_uid", "parent_row_uid", "dataset", "state_id", "state_index",
    "state_axis", "rep_index", "n_states", "n_reps_per_state", "split",
    "cv_fold", "Y_final", "task_type",
]
PRED_COLUMNS = ["pred_chemberta_regular", "pred_chemberta_augmented"]


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def datasets_in(variants_dir: Path) -> list[str]:
    return sorted(
        p.parent.name for p in variants_dir.glob("*/variants_nested.csv")
        if p.parent.name != "variants_a4_test"
    )


def discover_scorer(team_root: Path, explicit: str | None) -> Path:
    if explicit:
        path = Path(explicit)
        if not path.is_file():
            raise FileNotFoundError(f"score_variants_chemberta.py 없음: {path}")
        return path
    paths = sorted(set(walk_find(team_root, name="score_variants_chemberta.py")))
    if not paths:
        raise FileNotFoundError(f"{team_root} 아래 score_variants_chemberta.py 없음")
    groups: dict[str, list[Path]] = {}
    for path in paths:
        groups.setdefault(sha256(path), []).append(path)
    if len(groups) != 1:
        detail = "\n".join(f"  {sha256(p)[:12]}  {p}" for p in paths)
        raise RuntimeError(
            "내용이 다른 score_variants_chemberta.py가 여러 개다. --scorer로 지정한다:\n" + detail
        )
    chosen = min(paths, key=lambda p: (len(p.parts), str(p)))
    print(f"[확인] 기존 채점기: {chosen} (sha256={sha256(chosen)})")
    return chosen


def discover_checkpoint_root(team_root: Path, datasets: list[str], explicit: str | None) -> Path:
    if explicit:
        candidates = [Path(explicit)]
    else:
        roots = set()
        for complete in walk_find(team_root, name="complete.json"):
            if complete.parent.name in {"regular", "augmented"}:
                roots.add(complete.parent.parent.parent)
        candidates = sorted(roots)
    valid = []
    for root in candidates:
        missing = [
            f"{ds}/{version}" for ds in datasets for version in ("regular", "augmented")
            if not (root / ds / version / "complete.json").is_file()
        ]
        if not missing:
            valid.append(root)
    if len(valid) != 1:
        detail = "\n".join(f"  {p}" for p in candidates) or "  (후보 없음)"
        raise RuntimeError(
            "전체 대상 물성의 regular/augmented complete.json을 가진 체크포인트 루트를 "
            f"하나로 확정할 수 없다. --checkpoint-root로 지정한다.\n후보:\n{detail}"
        )
    print(f"[확인] 체크포인트 루트: {valid[0]}")
    return valid[0]


def _selected_split_signature(frame: pd.DataFrame) -> str:
    columns = [
        c for c in ("row_uid", "dataset", "split", "task_type", "parent_smiles", "cv_fold", "Y_final")
        if c in frame.columns
    ]
    value = frame[columns].sort_values("row_uid").to_csv(index=False)
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def find_splits(dataset: str, ids: set[str], search_roots: list[Path]) -> tuple[Path, pd.DataFrame]:
    candidates = []
    seen = set()
    for root in search_roots:
        if not root.exists():
            continue
        for path in walk_find(root, name="splits.csv"):
            if path.parent.name != dataset or path in seen:
                continue
            seen.add(path)
            try:
                frame = pd.read_csv(path, low_memory=False)
            except Exception:
                continue
            required = {"row_uid", "parent_smiles", "dataset", "split", "task_type"}
            if not required.issubset(frame.columns):
                continue
            selected = frame[frame["row_uid"].astype(str).isin(ids)].copy()
            if selected["row_uid"].astype(str).nunique() == len(ids):
                candidates.append((path, selected))
    if not candidates:
        raise FileNotFoundError(f"{dataset}: A-4 parent_row_uid를 모두 포함한 splits.csv 없음")
    signatures = {_selected_split_signature(frame) for _, frame in candidates}
    if len(signatures) != 1:
        detail = "\n".join(f"  {p}" for p, _ in candidates)
        raise RuntimeError(f"{dataset}: 내용이 다른 splits.csv 후보가 여러 개다:\n{detail}")
    candidates.sort(key=lambda item: (len(item[0].parts), str(item[0])))
    path, selected = candidates[0]
    selected = selected.copy()
    selected["row_uid"] = selected["row_uid"].astype(str)
    selected = selected[selected["row_uid"].isin(ids)]
    if selected["row_uid"].duplicated().any() or len(selected) != len(ids):
        raise ValueError(f"{dataset}: 임시 splits.csv의 row_uid가 일대일이 아님")
    print(f"[확인] {dataset} splits: {path}")
    return path, selected


def validate_existing(path: Path, input_rows: int) -> None:
    frame = pd.read_csv(path)
    if frame["variant_uid"].duplicated().any():
        raise ValueError(f"기존 결과 variant_uid 중복: {path}")
    if frame[PRED_COLUMNS].isna().any().any():
        raise ValueError(f"기존 결과 예측 결측: {path}")
    if len(frame) != input_rows:
        raise ValueError(f"기존 결과 행 수 불일치: {path} ({len(frame)} != {input_rows})")


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--variants-dir", required=True)
    ap.add_argument("--personal-root", required=True)
    ap.add_argument("--team-root", required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--work-dir", default="/content/a4_followup")
    ap.add_argument("--scorer", default=None)
    ap.add_argument("--checkpoint-root", default=None)
    ap.add_argument("--batch-size", type=int, default=128)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--splits-root", default=None,
                    help="splits.csv 를 찾을 루트. 지정하면 여기만 본다")
    ap.add_argument("--resume", action="store_true")
    args = ap.parse_args()

    variants_dir = Path(args.variants_dir)
    personal_root = Path(args.personal_root)
    team_root = Path(args.team_root)
    out_dir = Path(args.out_dir)
    work_dir = Path(args.work_dir)
    feed_dir, raw_dir = work_dir / "feed", work_dir / "raw"
    for path in (feed_dir, raw_dir, out_dir):
        path.mkdir(parents=True, exist_ok=True)

    splits_roots = ([Path(args.splits_root)] if args.splits_root
                    else [personal_root, team_root])
    print(f"[확인] splits 탐색 위치: {[str(p) for p in splits_roots]}")

    datasets = datasets_in(variants_dir)
    if not datasets:
        raise FileNotFoundError(f"대상 variants_nested.csv 없음: {variants_dir}")
    print(f"[확인] 대상 물성 {len(datasets)}종: {datasets}")
    scorer = discover_scorer(team_root, args.scorer)
    checkpoint_root = discover_checkpoint_root(team_root, datasets, args.checkpoint_root)

    local_scorer = work_dir / "score_variants_chemberta.py"
    shutil.copy2(scorer, local_scorer)
    repairs = scorer.parent / "dataset_repairs.py"
    if repairs.is_file():
        shutil.copy2(repairs, work_dir / repairs.name)
        print(f"[확인] dataset_repairs.py: {repairs}")
    else:
        (work_dir / "dataset_repairs.py").write_text(
            '"""A-4 임시 splits용 무동작 호환 모듈."""\n'
            "def apply_known_repairs(frame):\n    return frame, []\n",
            encoding="utf-8",
        )
        print("[확인] dataset_repairs.py가 없어 기존 노트북과 같은 무동작 호환 모듈 사용")

    pending = []
    inventory = []
    for dataset in datasets:
        nested_path = variants_dir / dataset / "variants_nested.csv"
        nested = pd.read_csv(nested_path, low_memory=False)
        missing = [c for c in KEEP_COLUMNS + ["variant_smiles"] if c not in nested.columns]
        if missing:
            raise ValueError(f"{nested_path}: 필수 열 없음 {missing}")
        if nested["variant_uid"].duplicated().any():
            raise ValueError(f"{nested_path}: variant_uid 중복")
        if set(nested["dataset"].dropna().astype(str)) != {dataset}:
            raise ValueError(f"{nested_path}: dataset 열과 폴더명 불일치")

        final_path = out_dir / dataset / "nested_predictions_chemberta.csv"
        if final_path.exists():
            if not args.resume:
                raise FileExistsError(f"기존 결과를 덮어쓰지 않음: {final_path} (--resume 사용)")
            validate_existing(final_path, len(nested))
            print(f"[resume] {dataset}: 검증된 기존 결과 건너뜀")
            inventory.append({"dataset": dataset, "n_input": len(nested), "status": "resumed"})
            continue

        target = feed_dir / dataset
        target.mkdir(parents=True, exist_ok=True)
        feed = nested.copy()
        feed["axis"] = feed["state_axis"]
        feed.to_csv(target / "variants.csv", index=False)
        ids = set(nested["parent_row_uid"].astype(str).unique())
        _, selected = find_splits(dataset, ids, splits_roots)
        selected.to_csv(target / "splits.csv", index=False)
        pending.append(dataset)
        inventory.append({"dataset": dataset, "n_input": len(nested), "status": "pending"})

    if pending:
        command = [
            sys.executable, str(local_scorer),
            "--variants-dir", str(feed_dir), "--splits-dir", str(feed_dir),
            "--checkpoint-root", str(checkpoint_root), "--out-dir", str(raw_dir),
            "--datasets", *pending, "--device", args.device,
            "--batch-size", str(args.batch_size), "--resume",
        ]
        print("[실행] 기존 채점기 원문:", " ".join(command))
        subprocess.run(command, check=True, cwd=work_dir)

    summary = []
    for item in inventory:
        dataset = item["dataset"]
        nested_path = variants_dir / dataset / "variants_nested.csv"
        final_dir = out_dir / dataset
        final_path = final_dir / "nested_predictions_chemberta.csv"
        if item["status"] == "resumed":
            frame = pd.read_csv(final_path)
        else:
            nested = pd.read_csv(nested_path, low_memory=False)
            raw = pd.read_csv(raw_dir / dataset / "variant_predictions_chemberta.csv")
            if raw["variant_uid"].duplicated().any():
                raise ValueError(f"{dataset}: 채점기 출력 variant_uid 중복")
            frame = nested[KEEP_COLUMNS].merge(
                raw[["variant_uid", *PRED_COLUMNS]], on="variant_uid", how="left",
                validate="one_to_one",
            )
            if frame["variant_uid"].duplicated().any():
                raise ValueError(f"{dataset}: 병합 후 variant_uid 중복")
            if frame[PRED_COLUMNS].isna().any().any():
                raise ValueError(f"{dataset}: 예측 병합 후 결측")
            if len(frame) != len(nested):
                raise ValueError(f"{dataset}: 입력/출력 행 수 불일치 {len(nested)} != {len(frame)}")
            final_dir.mkdir(parents=True, exist_ok=True)
            frame.to_csv(final_path, index=False)
            origin_src = raw_dir / dataset / "origin_predictions_chemberta.csv"
            origin_dst = final_dir / "origin_predictions_chemberta.csv"
            if origin_dst.exists():
                raise FileExistsError(f"기존 파일을 덮어쓰지 않음: {origin_dst}")
            shutil.copy2(origin_src, origin_dst)
        summary.append({
            "dataset": dataset, "n_rows": len(frame),
            "n_molecules": frame["parent_row_uid"].nunique(),
            "n_missing_predictions": int(frame[PRED_COLUMNS].isna().sum().sum()),
            "variant_uid_unique": not frame["variant_uid"].duplicated().any(),
            "status": item["status"],
        })

    summary_dir = out_dir / "_summary"
    summary_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(summary).to_csv(summary_dir / "scoring_summary.csv", index=False)
    (summary_dir / "resolved_inputs.json").write_text(json.dumps({
        "variants_dir": str(variants_dir), "scorer": str(scorer),
        "scorer_sha256": sha256(scorer), "checkpoint_root": str(checkpoint_root),
        "datasets": datasets,
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print(pd.DataFrame(summary).to_string(index=False))
    print(f"완료: {len(summary)}종, {sum(x['n_rows'] for x in summary):,}행")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing /content/a4_followup/score_nested_a4.py


In [ ]:
%%writefile /content/a4_followup/compute_formal_ab.py
#!/usr/bin/env python
"""Compute formal hierarchical A/B decomposition from A-4 ChemBERTa scores."""

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd


MODELS = {
    "regular": "pred_chemberta_regular",
    "augmented": "pred_chemberta_augmented",
}


def one_value(group: pd.DataFrame, column: str):
    values = group[column].drop_duplicates()
    if len(values) != 1:
        raise ValueError(
            f"{group['dataset'].iloc[0]}/{group['parent_row_uid'].iloc[0]}: "
            f"{column} 값이 하나가 아님: {values.tolist()}"
        )
    return values.iloc[0]


def decompose(group: pd.DataFrame, model_version: str, pred_col: str) -> dict:
    counts = group.groupby("state_id")["rep_index"].nunique()
    duplicate_rep = group.duplicated(["state_id", "rep_index"]).any()
    balanced = len(counts) >= 1 and counts.nunique() == 1 and not duplicate_rep
    state_means = group.groupby("state_id", sort=False)[pred_col].mean()
    state_vars = group.groupby("state_id", sort=False)[pred_col].var(ddof=0)
    a2 = float(state_vars.mean())
    b2 = float(state_means.var(ddof=0))
    total = float(group[pred_col].var(ddof=0))
    residual = total - a2 - b2
    n_reps = int(counts.iloc[0]) if counts.nunique() == 1 else pd.NA
    return {
        "dataset": one_value(group, "dataset"),
        "parent_row_uid": one_value(group, "parent_row_uid"),
        "split": one_value(group, "split"),
        "task_type": one_value(group, "task_type"),
        "model_version": model_version,
        "n_states": int(group["state_id"].nunique()),
        "n_reps_per_state": n_reps,
        "A2_formal": a2,
        "A_formal": float(np.sqrt(max(a2, 0.0))),
        "B2_formal": b2,
        "B_formal": float(np.sqrt(max(b2, 0.0))),
        "total_variance": total,
        "decomposition_residual": residual,
        "valid_decomposition": bool(balanced),
    }


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--scores-dir", required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--resume", action="store_true")
    args = ap.parse_args()
    scores_dir, out_dir = Path(args.scores_dir), Path(args.out_dir)
    sources = sorted(scores_dir.glob("*/nested_predictions_chemberta.csv"))
    if not sources:
        raise FileNotFoundError(f"채점 결과 없음: {scores_dir}")
    summaries = []
    for source in sources:
        dataset = source.parent.name
        target = out_dir / dataset / "formal_ab_decomposition.csv"
        if target.exists():
            if not args.resume:
                raise FileExistsError(f"기존 결과를 덮어쓰지 않음: {target} (--resume 사용)")
            old = pd.read_csv(target)
            summaries.append({"dataset": dataset, "n_rows": len(old), "status": "resumed"})
            print(f"[resume] {dataset}: 기존 결과 건너뜀")
            continue
        frame = pd.read_csv(source, low_memory=False)
        required = {
            "dataset", "parent_row_uid", "state_id", "rep_index", "split", "task_type", *MODELS.values()
        }
        missing = sorted(required - set(frame.columns))
        if missing:
            raise ValueError(f"{source}: 필수 열 없음 {missing}")
        rows = []
        for _, group in frame.groupby("parent_row_uid", sort=False):
            for version, pred_col in MODELS.items():
                rows.append(decompose(group, version, pred_col))
        result = pd.DataFrame(rows)
        target.parent.mkdir(parents=True, exist_ok=True)
        result.to_csv(target, index=False)
        valid = result[result["valid_decomposition"]]
        max_residual = float(valid["decomposition_residual"].abs().max()) if len(valid) else np.nan
        summaries.append({
            "dataset": dataset, "n_rows": len(result),
            "n_molecules": result["parent_row_uid"].nunique(),
            "n_invalid": int((~result["valid_decomposition"]).sum()),
            "max_abs_residual_valid": max_residual, "status": "computed",
        })
        print(
            f"{dataset}: {len(result):,}행, invalid={int((~result.valid_decomposition).sum()):,}, "
            f"유효 분해 잔차 최대={max_residual:.3e}"
        )
    summary_dir = out_dir / "_summary"
    summary_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(summaries).to_csv(summary_dir / "formal_ab_summary.csv", index=False)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing /content/a4_followup/compute_formal_ab.py


In [ ]:
%%writefile /content/a4_followup/compare_approx_formal.py
#!/usr/bin/env python
"""Audit variants_v2 provenance, match approximate/formal A/B, and compare them."""

from __future__ import annotations

import argparse
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd



# ---------------------------------------------------------------------------
# 구글 드라이브 마운트에서 Path.rglob 는 바로가기를 통과하지 못해 빈 결과를 준다.
# iterdir 재귀로 같은 일을 한다. 이 파일의 모든 탐색은 이 함수를 쓴다.
# ---------------------------------------------------------------------------
_SKIP_DIRS = {"__pycache__", ".ipynb_checkpoints", ".git"}


def walk_find(root, name=None, suffix=None, max_depth=8):
    """root 아래 파일을 재귀로 찾는다. name 정확 일치 또는 suffix 로 거른다."""
    found = []

    def rec(d, depth):
        if depth > max_depth:
            return
        try:
            entries = list(d.iterdir())
        except Exception:
            return
        for e in entries:
            try:
                is_dir = e.is_dir()
            except Exception:
                continue
            if is_dir:
                if e.name not in _SKIP_DIRS:
                    rec(e, depth + 1)
            elif name is not None:
                if e.name == name:
                    found.append(e)
            elif suffix is not None:
                if e.suffix.lower() == suffix:
                    found.append(e)
            else:
                found.append(e)

    rec(Path(root), 0)
    return found

KEYS = ["dataset", "parent_row_uid", "split", "model_version"]
TEXT_SUFFIXES = {".py", ".json", ".md", ".txt", ".ipynb", ".yaml", ".yml"}
A_NAMES = ["A_approx", "A", "A_smiles", "A_chemberta", "approx_A"]
B_NAMES = ["B_approx", "B", "B_state", "B_chemberta", "approx_B"]


def read_small_text(path: Path) -> str:
    try:
        if path.stat().st_size > 5_000_000:
            return ""
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""


def provenance_evidence(candidate: Path, roots: list[Path]) -> list[str]:
    evidence = []
    if "variants_v2" in str(candidate).lower().replace("\\", "/"):
        evidence.append(f"경로에 variants_v2 명시: {candidate}")
    names = {candidate.name, candidate.stem}
    for root in roots:
        if not root.exists():
            continue
        for path in walk_find(root):
            if not path.is_file() or path.suffix.lower() not in TEXT_SUFFIXES:
                continue
            text = read_small_text(path)
            low = text.lower()
            if "variants_v2" not in low:
                continue
            related = path.parent == candidate.parent or any(name in text for name in names)
            if related:
                evidence.append(f"{path}: variants_v2와 후보 파일/동일 폴더 연결")
    return sorted(set(evidence))


def header(path: Path) -> list[str]:
    try:
        return list(pd.read_csv(path, nrows=0).columns)
    except Exception:
        return []


def find_candidates(roots: list[Path]) -> list[dict]:
    rows, seen = [], set()
    for root in roots:
        if not root.exists():
            continue
        for path in walk_find(root, suffix=".csv"):
            if path in seen:
                continue
            seen.add(path)
            cols = header(path)
            normalized = {"parent_row_uid" if c == "row_uid" else c for c in cols}
            if not {"dataset", "parent_row_uid"}.issubset(normalized):
                continue
            signal_cols = [c for c in cols if re.search(r"(^|_)(a|b)(_|$)|variance", c, re.I)]
            if not signal_cols:
                continue
            evidence = provenance_evidence(path, roots)
            rows.append({
                "path": str(path), "columns": " | ".join(cols),
                "signal_columns": " | ".join(signal_cols),
                "variants_v2_confirmed": bool(evidence),
                "provenance_evidence": " || ".join(evidence),
            })
    return rows


def pick_column(columns: list[str], requested: str | None, names: list[str], label: str) -> str:
    if requested:
        if requested not in columns:
            raise ValueError(f"지정한 {label} 열 없음: {requested}")
        return requested
    hits = [name for name in names if name in columns]
    if len(hits) != 1:
        raise RuntimeError(f"{label} 열을 하나로 확정할 수 없음. 후보={hits}, 전체={columns}")
    return hits[0]


def normalize_approx(path: Path, a_col: str | None, b_col: str | None) -> pd.DataFrame:
    frame = pd.read_csv(path, low_memory=False)
    if "parent_row_uid" not in frame.columns and "row_uid" in frame.columns:
        frame = frame.rename(columns={"row_uid": "parent_row_uid"})
    missing_keys = [c for c in KEYS if c not in frame.columns]
    if missing_keys:
        raise RuntimeError(
            f"근사 신호가 long 형식 키를 갖지 않는다: {missing_keys}. "
            "열을 추측해 wide 형식을 변환하지 않는다."
        )
    a = pick_column(list(frame.columns), a_col, A_NAMES, "근사 A")
    b = pick_column(list(frame.columns), b_col, B_NAMES, "근사 B")
    out = frame[KEYS + [a, b]].rename(columns={a: "A_approx", b: "B_approx"})
    if out.duplicated(KEYS).any():
        raise ValueError("근사 신호 연결 키가 중복됨")
    return out


def load_formal(formal_dir: Path) -> pd.DataFrame:
    paths = sorted(formal_dir.glob("*/formal_ab_decomposition.csv"))
    if not paths:
        raise FileNotFoundError(f"정식 분해 결과 없음: {formal_dir}")
    frame = pd.concat((pd.read_csv(path) for path in paths), ignore_index=True)
    frame = frame[frame["valid_decomposition"].astype(bool)].copy()
    if frame.duplicated(KEYS).any():
        raise ValueError("정식 분해 연결 키가 중복됨")
    return frame


def summarize(matched: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (dataset, split, model), group in matched.groupby(["dataset", "split", "model_version"]):
        for signal in ("A", "B"):
            approx, formal = f"{signal}_approx", f"{signal}_formal"
            pair = group[[approx, formal]].dropna()
            rows.append({
                "dataset": dataset, "split": split, "model_version": model,
                "signal": signal, "n": len(pair),
                "spearman": pair[approx].corr(pair[formal], method="spearman") if len(pair) >= 2 else float("nan"),
                "approx_mean": pair[approx].mean(), "approx_median": pair[approx].median(),
                "formal_mean": pair[formal].mean(), "formal_median": pair[formal].median(),
            })
    result = pd.DataFrame(rows)
    result["split_order"] = result["split"].map({"test": 0, "meta": 1}).fillna(9)
    return result.sort_values(["split_order", "dataset", "model_version", "signal"]).drop(columns="split_order")


def figures(matched: pd.DataFrame, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    test = matched[matched["split"] == "test"]
    for (dataset, model), group in test.groupby(["dataset", "model_version"]):
        fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
        for ax, signal in zip(axes, ("A", "B")):
            x, y = f"{signal}_approx", f"{signal}_formal"
            pair = group[[x, y]].dropna()
            rho = pair[x].corr(pair[y], method="spearman") if len(pair) >= 2 else float("nan")
            ax.scatter(pair[x], pair[y], s=14, alpha=0.55, edgecolors="none")
            ax.set_xlabel(f"Approximate {signal} (variants_v2)")
            ax.set_ylabel(f"Formal {signal} (A-4)")
            ax.set_title(f"{signal}: n={len(pair)}, Spearman={rho:.3f}")
            ax.grid(alpha=0.2)
        fig.suptitle(f"{dataset} | {model} | test")
        fig.tight_layout()
        safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", dataset)
        fig.savefig(out_dir / f"{safe}__{model}__test.png", dpi=180, bbox_inches="tight")
        plt.close(fig)


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--formal-dir", required=True)
    ap.add_argument("--personal-root", required=True)
    ap.add_argument("--team-root", required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--approx-path", default=None)
    ap.add_argument("--approx-a-col", default=None)
    ap.add_argument("--approx-b-col", default=None)
    ap.add_argument("--resume", action="store_true")
    args = ap.parse_args()
    formal_dir, out_dir = Path(args.formal_dir), Path(args.out_dir)
    roots = [Path(args.team_root), Path(args.personal_root)]
    out_dir.mkdir(parents=True, exist_ok=True)
    audit_path = out_dir / "approx_source_audit.csv"
    audit = pd.DataFrame(find_candidates(roots))
    audit.to_csv(audit_path, index=False)
    print(f"[감사] 근사 신호 후보 {len(audit)}개 -> {audit_path}")

    if args.approx_path:
        source = Path(args.approx_path)
        row = audit[audit["path"] == str(source)]
        if row.empty or not bool(row.iloc[0]["variants_v2_confirmed"]):
            raise RuntimeError("지정한 근사 신호의 variants_v2 출처를 확인할 수 없어 비교하지 않음")
    else:
        confirmed = audit[audit["variants_v2_confirmed"]] if len(audit) else audit
        if len(confirmed) != 1:
            print(confirmed[["path", "signal_columns", "provenance_evidence"]].to_string(index=False) if len(confirmed) else "확인된 후보 없음")
            raise RuntimeError(
                "variants_v2 기반 근사 A/B 결과를 하나로 확정할 수 없다. "
                "임의의 variants_role3 신호는 사용하지 않았다. 감사표를 확인한다."
            )
        source = Path(confirmed.iloc[0]["path"])

    evidence = provenance_evidence(source, roots)
    if not evidence:
        raise RuntimeError("variants_v2 출처 근거가 없어 비교 중단")
    print(f"[확인] 근사 신호: {source}")
    for item in evidence:
        print("  -", item)

    matched_path = out_dir / "matched_approx_formal.csv"
    summary_path = out_dir / "correlation_summary.csv"
    if (matched_path.exists() or summary_path.exists()) and not args.resume:
        raise FileExistsError("기존 비교 결과를 덮어쓰지 않음 (--resume 사용)")
    if args.resume and matched_path.exists() and summary_path.exists():
        print("[resume] 기존 비교 결과 건너뜀")
        print(pd.read_csv(summary_path).to_string(index=False))
        return 0

    approx = normalize_approx(source, args.approx_a_col, args.approx_b_col)
    formal = load_formal(formal_dir)
    matched = formal.merge(approx, on=KEYS, how="inner", validate="one_to_one")
    if matched.empty:
        raise RuntimeError("연결 키가 일치하는 근사/정식 표본이 없음")
    matched.to_csv(matched_path, index=False)
    summary = summarize(matched)
    summary.to_csv(summary_path, index=False)
    figures(matched, out_dir / "figures")
    (out_dir / "comparison_metadata.json").write_text(json.dumps({
        "approx_source": str(source), "variants_v2_evidence": evidence,
        "join_keys": KEYS, "summary_keeps_meta_and_test_separate": True,
        "figures_split": "test",
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print("\n[test Spearman]")
    print(summary[summary["split"] == "test"].to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

Writing /content/a4_followup/compare_approx_formal.py


In [ ]:
# ============================================================================
# [셀 7] 스크립트 확인
# ============================================================================
for n in ("score_nested_a4.py", "compute_formal_ab.py", "compare_approx_formal.py"):
    p = WORK_DIR / n
    t = p.read_text(encoding="utf-8")
    print(f"{n:26s} {len(t.splitlines()):>4}줄 | 남은 rglob 호출 {t.count('.rglob(')}")

score_nested_a4.py          330줄 | 남은 rglob 호출 0
compute_formal_ab.py        112줄 | 남은 rglob 호출 0
compare_approx_formal.py    272줄 | 남은 rglob 호출 0


## 3단계 — 채점

담당4의 `score_variants_chemberta.py` 원문을 복사해 호출한다.
`--resume` 이 있어 끊겨도 이어받는다.

In [ ]:
# ============================================================================
# [셀 8] A-4 전체 물성 ChemBERTa 채점
# ============================================================================
assert SCORER_PATH is not None, "SCORER_PATH 미확정 — 셀 3 출력 확인"
assert CHECKPOINT_ROOT is not None, "CHECKPOINT_ROOT 미확정 — 셀 3 아래 설명대로 직접 지정"

cmd = [sys.executable, str(WORK_DIR / "score_nested_a4.py"),
       "--variants-dir", str(VARIANTS_DIR),
       "--personal-root", str(PERSONAL_ROOT),
       "--team-root", str(PROJECT_ROOT),
       "--splits-root", str(SPLITS_ROOT),
       "--scorer", str(SCORER_PATH),
       "--checkpoint-root", str(CHECKPOINT_ROOT),
       "--out-dir", str(SCORES_DIR),
       "--work-dir", str(WORK_DIR),
       "--device", "cuda", "--batch-size", "128", "--resume"]
print(" ".join(cmd), "\n")
subprocess.run(cmd, check=True)

/usr/bin/python3 /content/a4_followup/score_nested_a4.py --variants-dir /content/drive/MyDrive/Conference_2026/Yoonsoo/variants_a4_full --personal-root /content/drive/MyDrive/Conference_2026/Yoonsoo --team-root /content/drive/MyDrive/Conference_2026 --splits-root /content/drive/MyDrive/Conference_2026/Juhyeong/data/processed/pipeline_yoonsoo --scorer /content/drive/MyDrive/Conference_2026/Main/scripts_role4/score_variants_chemberta.py --checkpoint-root /content/drive/MyDrive/Conference_2026/Jiye/checkpoints/chemberta_seed_42/checkpoints --out-dir /content/drive/MyDrive/Conference_2026/Yoonsoo/scores_a4_full --work-dir /content/a4_followup --device cuda --batch-size 128 --resume 



CompletedProcess(args=['/usr/bin/python3', '/content/a4_followup/score_nested_a4.py', '--variants-dir', '/content/drive/MyDrive/Conference_2026/Yoonsoo/variants_a4_full', '--personal-root', '/content/drive/MyDrive/Conference_2026/Yoonsoo', '--team-root', '/content/drive/MyDrive/Conference_2026', '--splits-root', '/content/drive/MyDrive/Conference_2026/Juhyeong/data/processed/pipeline_yoonsoo', '--scorer', '/content/drive/MyDrive/Conference_2026/Main/scripts_role4/score_variants_chemberta.py', '--checkpoint-root', '/content/drive/MyDrive/Conference_2026/Jiye/checkpoints/chemberta_seed_42/checkpoints', '--out-dir', '/content/drive/MyDrive/Conference_2026/Yoonsoo/scores_a4_full', '--work-dir', '/content/a4_followup', '--device', 'cuda', '--batch-size', '128', '--resume'], returncode=0)

compare_approx_formal.py 를 만드는 %%writefile 셀(셀9) 폐기

In [10]:
# ============================================================================
# [셀 10] 정식 B vs 운영 파이프라인 근사 B — 신호 수준 대조
# ----------------------------------------------------------------------------
# compare_approx_formal.py 는 폐기했다. 그 스크립트는 근사 신호가
# variants_v2 경로에 long 형식으로 있을 것을 전제하는데, A-3 산출물은
# scores_v2 의 wide 형식이다. 우회하지 않고 여기서 직접 잇는다.
#
# 셀 12 의 근사 B 는 중첩 데이터 내부(rep_index 0)에서 만든 값이다.
# 여기서는 실제 파이프라인이 쓰는 값과 정식 B 를 맞대본다.
# ============================================================================
import numpy as np
from scipy.stats import spearmanr

V2_EVAL = PROJECT_ROOT / "Yoonsoo" / "scores_v2" / "evaluation"
assert V2_EVAL.is_dir(), f"A-3 채점 결과 없음: {V2_EVAL}"

# 폐기한 스크립트가 남긴 감사표 정리
stale = COMPARE_DIR / "approx_source_audit.csv"
if stale.exists():
    stale.unlink(); print(f"제거: {stale.name}")

# 근사 B 후보를 눈으로 확인하고 고른다 (자동 추측하지 않는다)
probe = pd.read_csv(next(V2_EVAL.glob("*/evaluation_signals.csv")), nrows=0)
cands = [c for c in probe.columns if "cb_augmented" in c and ("__B" in c or c.startswith("cond_B"))]
print("근사 B 후보:", cands)

B_PIPE = "cond_B__cb_augmented__std"   # 셀 13 파이프 arm 과 같은 열로 고정
assert B_PIPE in probe.columns, f"{B_PIPE} 없음. 위 후보에서 고른다"

rows = []
for ds in datasets:
    f = DECOMP_DIR / ds / "formal_ab_decomposition.csv"
    v = V2_EVAL / ds / "evaluation_signals.csv"
    if not (f.is_file() and v.is_file()):
        print(f"[건너뜀] {ds}"); continue

    fm = pd.read_csv(f)
    fm = fm[fm.valid_decomposition.astype(bool) & (fm.model_version == "augmented")]
    fm = fm[["dataset", "parent_row_uid", "split", "B_formal", "A_formal"]]

    pv = pd.read_csv(v, low_memory=False)[["row_uid", B_PIPE]].rename(
        columns={"row_uid": "parent_row_uid", B_PIPE: "B_pipe"})

    m = fm.merge(pv, on="parent_row_uid", how="inner").dropna(subset=["B_formal", "B_pipe"])
    if len(m) < 20:
        print(f"[건너뜀] {ds}: 병합 {len(m)}행"); continue

    te = m[m.split == "test"]
    if len(te) < 20:
        te = m
    rows.append({
        "dataset": ds, "n": len(te),
        "spearman": spearmanr(te.B_formal, te.B_pipe).statistic,
        "정식_중앙": te.B_formal.median(),
        "파이프_중앙": te.B_pipe.median(),
        "비": te.B_formal.median() / te.B_pipe.median() if te.B_pipe.median() else np.nan,
    })

sig = pd.DataFrame(rows)
sig.to_csv(COMPARE_DIR / "signal_formal_vs_pipeline_B.csv", index=False)
print(f"\n물성 {len(sig)}종 (test 분할, cb_augmented)\n")
print(sig.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\n중앙 Spearman {sig.spearman.median():.3f}"
      f"  |  정식/파이프 중앙비 {sig['비'].median():.3f}"
      f"  (범위 {sig['비'].min():.2f}~{sig['비'].max():.2f})")

근사 B 후보: ['axis__cb_augmented__B1_tautomer', 'axis__cb_augmented__B1_protonation', 'axis__cb_augmented__B3_stereo', 'axis__cb_augmented__B_combined', 'axis__cb_augmented__B1_tautomer__pct', 'axis__cb_augmented__B1_protonation__pct', 'axis__cb_augmented__B3_stereo__pct', 'axis__cb_augmented__B_combined__pct', 'cond_B__cb_augmented__std', 'cond_B__cb_augmented__std__pct', 'rich__cb_augmented__B__std', 'rich__cb_augmented__B__max_dev', 'rich__cb_augmented__B__shift', 'rich__cb_augmented__B__rel_std', 'rich__cb_augmented__B__flip', 'rich__cb_augmented__B__std__pct', 'rich__cb_augmented__B__max_dev__pct', 'rich__cb_augmented__B__shift__pct', 'rich__cb_augmented__B__rel_std__pct', 'rich__cb_augmented__B__flip__pct']

물성 22종 (test 분할, cb_augmented)

                       dataset    n  spearman  정식_중앙  파이프_중앙      비
                          ames  534    0.7456 0.0302  0.0188 1.6073
                   bbb_martins  187    0.7967 0.0152  0.0124 1.2296
            bioavailability_ma   63    0.39

In [ ]:
# ============================================================================
# [셀 11] 요약
# ============================================================================
score_summary = pd.read_csv(SCORES_DIR / "_summary" / "scoring_summary.csv")
formal = pd.read_csv(DECOMP_DIR / "_summary" / "formal_ab_summary.csv")

print("채점 :", SCORES_DIR)
print("분해 :", DECOMP_DIR)
print("비교 :", COMPARE_DIR)
print(f"\n물성 {score_summary['dataset'].nunique()}종, "
      f"변형 {int(score_summary['n_rows'].sum()):,}행\n")
print(formal.to_string(index=False))

corr_path = COMPARE_DIR / "correlation_summary.csv"
if corr_path.exists():
    corr = pd.read_csv(corr_path)
    print("\n=== 근사–정식 Spearman (test) ===")
    print(corr[corr["split"] == "test"].to_string(index=False))
else:
    audit = COMPARE_DIR / "approx_source_audit.csv"
    print("\n비교 미생성. 감사표:", audit)
    if audit.exists():
        print(pd.read_csv(audit).to_string(index=False))

채점 : /content/drive/MyDrive/Conference_2026/Yoonsoo/scores_a4_full
분해 : /content/drive/MyDrive/Conference_2026/Yoonsoo/a4_decomposition_full
비교 : /content/drive/MyDrive/Conference_2026/Yoonsoo/a4_comparison_full

물성 22종, 변형 1,384,692행

                       dataset  n_rows  n_molecules  n_invalid  max_abs_residual_valid   status
                          ames    2116         1058          0            9.020562e-17 computed
                   bbb_martins     756          378          0            1.176359e-17 computed
            bioavailability_ma     254          127          0            2.466560e-18 computed
                    caco2_wang     352          176          0            1.474515e-17 computed
       clearance_hepatocyte_az     404          202          0            1.054712e-14 computed
        clearance_microsome_az     434          217          0            4.884981e-15 computed
cyp2c9_substrate_carbonmangels     258          129          0            5.204170e-18 compu

In [8]:
# ============================================================================
# [셀 12] 5단계 — 같은 데이터에서 근사 정의 vs 정식 정의
# ----------------------------------------------------------------------------
# 근사 정의도 중첩 데이터의 부분집합으로 계산된다. 추가 채점이 필요 없다.
#
#   정식   A² = 상태 내부 분산을 상태들에 대해 평균
#          B² = 상태별 평균들 사이의 분산
#
#   근사   A² = 원본 상태(parent) 안에서의 분산          <- 한 상태만 본다
#          B² = 상태마다 표기 하나(rep_index==0)로 잰 분산 <- 표기 하나씩
#
# 분자·모델·변형이 완전히 같고 계산 방식만 다르므로 정의 차이가 순수하게 분리된다.
# ============================================================================
import numpy as np
from scipy.stats import spearmanr

PRED = {"regular": "pred_chemberta_regular", "augmented": "pred_chemberta_augmented"}
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for ds in datasets:
    scored = pd.read_csv(SCORES_DIR / ds / "nested_predictions_chemberta.csv", low_memory=False)
    for version, col in PRED.items():
        for uid, g in scored.groupby("parent_row_uid", sort=False):
            # --- 근사 ---
            parent = g[g.state_axis == "parent"]
            a2_ap = float(parent[col].var(ddof=0)) if len(parent) > 1 else np.nan
            one_rep = g[g.rep_index == 0]
            b2_ap = float(one_rep[col].var(ddof=0)) if len(one_rep) > 1 else np.nan
            rows.append({
                "dataset": ds, "parent_row_uid": uid, "model_version": version,
                "split": g["split"].iloc[0],
                "A2_approx": a2_ap, "B2_approx": b2_ap,
                "A_approx": np.sqrt(a2_ap) if a2_ap == a2_ap else np.nan,
                "B_approx": np.sqrt(b2_ap) if b2_ap == b2_ap else np.nan,
            })

approx = pd.DataFrame(rows)

formal = pd.concat(
    [pd.read_csv(DECOMP_DIR / ds / "formal_ab_decomposition.csv") for ds in datasets],
    ignore_index=True)

KEY = ["dataset", "parent_row_uid", "model_version"]
both = formal.merge(approx.drop(columns=["split"]), on=KEY, how="inner", validate="one_to_one")
both.to_csv(COMPARE_DIR / "formal_vs_approx_same_data.csv", index=False)
print(f"대조 {len(both):,}행 ({both.parent_row_uid.nunique():,}분자 × {both.model_version.nunique()}모델)\n")

# --- 물성 × 모델별 상관 ---
out = []
for (ds, ver, sp), g in both.groupby(["dataset", "model_version", "split"]):
    for axis in ("A", "B"):
        x, y = g[f"{axis}_approx"], g[f"{axis}_formal"]
        ok = x.notna() & y.notna()
        if ok.sum() < 10:
            continue
        rho, p = spearmanr(x[ok], y[ok])
        out.append({"dataset": ds, "model": ver, "split": sp, "axis": axis,
                    "n": int(ok.sum()), "spearman": round(rho, 4),
                    "p": f"{p:.2e}",
                    "근사_중앙": round(x[ok].median(), 5),
                    "정식_중앙": round(y[ok].median(), 5)})

corr = pd.DataFrame(out)
corr.to_csv(COMPARE_DIR / "formal_vs_approx_correlation.csv", index=False)

print("=== B축 (1차 가설의 축) ===")
print(corr[corr.axis == "B"].to_string(index=False))
print("\n=== A축 ===")
print(corr[corr.axis == "A"].to_string(index=False))
print("\n저장:", COMPARE_DIR / "formal_vs_approx_correlation.csv")

대조 29,008행 (14,504분자 × 2모델)

=== B축 (1차 가설의 축) ===
                       dataset     model split axis    n  spearman         p   근사_중앙   정식_중앙
                          ames augmented  meta    B  524    0.7766 8.19e-107 0.04596 0.02550
                          ames augmented  test    B  534    0.7961 3.84e-118 0.05367 0.03016
                          ames   regular  meta    B  524    0.6287  5.40e-59 0.03202 0.01484
                          ames   regular  test    B  534    0.6810  5.31e-74 0.03046 0.01539
                   bbb_martins augmented  meta    B  191    0.7888  8.37e-42 0.03249 0.01327
                   bbb_martins augmented  test    B  187    0.8064  4.58e-44 0.03651 0.01520
                   bbb_martins   regular  meta    B  191    0.4917  5.06e-13 0.01402 0.00548
                   bbb_martins   regular  test    B  187    0.4974  4.43e-13 0.01410 0.00554
            bioavailability_ma augmented  meta    B   64    0.5294  6.85e-06 0.03612 0.01209
            bioavai

In [11]:
# ============================================================================
# [셀 13] 제거 실험 — 근사 B vs 정식 B (22종)
# ----------------------------------------------------------------------------
# run_preregistered_ablation.py 의 정의를 그대로 따른다.
#   대상값  meta 의 abs_error_fp 순위 / 결합기 Ridge(alpha=1.0)
#   라벨    분류: 0.5 기준 오분류 / 회귀: 오차 상위 20%
#   지표    average_precision_score, 정규화 AURC
#
# [교정] 이전 판은 근사 B 를 담당4의 evaluation_signals 에서 가져왔다.
# 그 값은 variants_role3 기반(A축 10, pH 6.4~8.4, B3 분자당 1개)인데
# 정식 B 는 중첩 변형 기반(pH 5.0~9.0, B3 부분 제거)이라
# 정의 차이와 상태집합 차이가 섞여 있었다.
# 중첩 데이터 안에 근사 정의가 그대로 있다(상태당 표기 1개 = rep_index 0).
# 그것을 쓰면 상태집합이 같아져 정의 차이만 남는다.
# 대조용으로 기존 파이프라인 근사 arm 도 함께 돌린다.
# ============================================================================
import numpy as np
from scipy.stats import rankdata, wilcoxon
from sklearn.linear_model import Ridge
from sklearn.metrics import average_precision_score

EVAL  = PROJECT_ROOT / "Main" / "evaluation_role4"        # 읽기만 한다
BASE  = ["base__ad_knn__pct", "base__ad_density__pct",
         "base__conformal_cb__pct", "base__conformal_fp__pct"]
A_COL = "axis__cb_augmented__A__pct"

def naurc(s, e):
    f = lambda x: float(np.mean(np.cumsum(e[np.argsort(x, kind="stable")]) / np.arange(1, len(e)+1)))
    o, r = f(e), float(np.mean(e))
    return np.nan if r - o < 1e-12 else (f(s) - o) / (r - o)

# 파이프라인이 cb_augmented 를 쓰므로 근사·정식 모두 augmented 로 맞춘다
fb = (both[both.model_version == "augmented"]
      [["dataset", "parent_row_uid", "B_approx", "B_formal"]]
      .dropna(subset=["B_approx", "B_formal"])
      .rename(columns={"parent_row_uid": "row_uid"}))
assert len(fb), "both 에 augmented 행이 없다 — 셀 12 를 먼저 실행한다"

rows = []
for ds in datasets:
    m = pd.read_csv(EVAL / ds / "evaluation_signals.csv", low_memory=False)
    m = m.merge(fb[fb.dataset == ds].drop(columns="dataset"), on="row_uid", how="inner")
    if len(m) < 40:
        print(f"[건너뜀] {ds}: 병합 후 {len(m)}행"); continue

    # B 특성 백분위를 같은 부분집합에서 다시 매긴다 (세 arm 동일 조건)
    for src, name in [("cond_B__fp_primary__std",   "fpB"),
                      ("cond_B__cb_augmented__std", "cbB_pipe"),    # 파이프라인 근사
                      ("B_approx",                  "cbB_approx"),  # 중첩 근사
                      ("B_formal",                  "cbB_formal")]:
        m[name + "__pct"] = rankdata(m[src]) / len(m)

    me, te = m[m.split == "meta"], m[m.split == "test"]
    if len(me) < 20 or len(te) < 20:
        print(f"[건너뜀] {ds}: meta {len(me)} / test {len(te)}"); continue

    err = te.abs_error_fp.to_numpy(float)
    tgt = rankdata(me.abs_error_fp) / len(me)
    lab = (((te.pred_fp_primary.to_numpy(float) >= .5).astype(int)
            != pd.to_numeric(te.Y_final).astype(int)).astype(int)
           if m.task_type.iloc[0] == "classification"
           else (err >= np.quantile(err, .8)).astype(int))

    CFG = {"기준":            BASE,
           "기준+A":          BASE + [A_COL],
           "기준+B(파이프)":  BASE + ["fpB__pct", "cbB_pipe__pct"],
           "기준+B(근사)":    BASE + ["fpB__pct", "cbB_approx__pct"],
           "기준+B(정식)":    BASE + ["fpB__pct", "cbB_formal__pct"]}

    r = {"dataset": ds, "n_meta": len(me), "n_test": len(te)}
    for n, f in CFG.items():
        u = [c for c in f if c in m and m[c].nunique() > 1]
        s = Ridge(alpha=1.0).fit(me[u].to_numpy(float), tgt).predict(te[u].to_numpy(float))
        r[f"auprc__{n}"] = average_precision_score(lab, s) if lab.min() != lab.max() else np.nan
        r[f"aurc__{n}"]  = naurc(s, err)
    rows.append(r)

ab = pd.DataFrame(rows)
ab.to_csv(COMPARE_DIR / "ablation_formal_vs_approx_B.csv", index=False)
print(f"물성 {len(ab)}종\n")

ORDER = ["기준", "기준+A", "기준+B(파이프)", "기준+B(근사)", "기준+B(정식)"]
print(f"{'구성':16s}{'AUPRC':>9s}{'AURC':>9s}")
for n in ORDER:
    print(f"{n:16s}{ab[f'auprc__{n}'].mean():9.4f}{ab[f'aurc__{n}'].mean():9.4f}")

print("\n=== 기준 대비 (AURC 는 낮을수록 좋으므로 부호를 뒤집음) ===")
for n in ORDER[1:]:
    for met, col, sign in (("AUPRC", "auprc", 1), ("AURC", "aurc", -1)):
        d = ((ab[f"{col}__{n}"] - ab[f"{col}__기준"]) * sign).dropna()
        try:
            p = wilcoxon(d)[1]
        except Exception:
            p = np.nan
        print(f"  {n:16s} {met:5s} {d.mean():+.4f}  개선 {int((d>0).sum())}/{len(d)}  p={p:.3f}")

def gain(n, col, sign):
    return ((ab[f"{col}__{n}"] - ab[f"{col}__기준"]) * sign).mean()

print("\n=== 정식/근사 비 ===")
print("  (a) 중첩 근사 기준 — 상태집합 동일. 정의 차이만 본다")
print("  (b) 파이프라인 근사 기준 — 구판 34%/104% 가 여기서 재현돼야 한다")
for met, col, sign in (("AUPRC", "auprc", 1), ("AURC", "aurc", -1)):
    fo = gain("기준+B(정식)", col, sign)
    ap = gain("기준+B(근사)", col, sign)
    pi = gain("기준+B(파이프)", col, sign)
    ra = f"{fo/ap*100:.0f}%" if abs(ap) > 1e-9 else "n/a"
    rp = f"{fo/pi*100:.0f}%" if abs(pi) > 1e-9 else "n/a"
    print(f"  {met:5s} 정식 {fo:+.4f} | (a) 근사 {ap:+.4f} -> {ra:>6s}"
          f" | (b) 파이프 {pi:+.4f} -> {rp:>6s}")

print("\n=== 상태집합 차이만의 효과 (근사 정의 고정, 데이터만 교체) ===")
for met, col, sign in (("AUPRC", "auprc", 1), ("AURC", "aurc", -1)):
    print(f"  {met:5s} 파이프 {gain('기준+B(파이프)', col, sign):+.4f}"
          f" -> 중첩 {gain('기준+B(근사)', col, sign):+.4f}")

물성 22종

구성                  AUPRC     AURC
기준                 0.3093   0.5983
기준+A               0.3207   0.5986
기준+B(파이프)          0.3160   0.5744
기준+B(근사)           0.3251   0.5882
기준+B(정식)           0.3116   0.5734

=== 기준 대비 (AURC 는 낮을수록 좋으므로 부호를 뒤집음) ===
  기준+A             AUPRC +0.0113  개선 13/22  p=0.235
  기준+A             AURC  -0.0003  개선 14/22  p=0.371
  기준+B(파이프)        AUPRC +0.0066  개선 12/22  p=0.322
  기준+B(파이프)        AURC  +0.0239  개선 13/22  p=0.114
  기준+B(근사)         AUPRC +0.0158  개선 13/22  p=0.105
  기준+B(근사)         AURC  +0.0101  개선 15/22  p=0.235
  기준+B(정식)         AUPRC +0.0023  개선 14/22  p=0.824
  기준+B(정식)         AURC  +0.0249  개선 15/22  p=0.042

=== 정식/근사 비 ===
  (a) 중첩 근사 기준 — 상태집합 동일. 정의 차이만 본다
  (b) 파이프라인 근사 기준 — 구판 34%/104% 가 여기서 재현돼야 한다
  AUPRC 정식 +0.0023 | (a) 근사 +0.0158 ->    14% | (b) 파이프 +0.0066 ->    34%
  AURC  정식 +0.0249 | (a) 근사 +0.0101 ->   247% | (b) 파이프 +0.0239 ->   104%

=== 상태집합 차이만의 효과 (근사 정의 고정, 데이터만 교체) ===
  AUPRC 파이프 +0.0066 -> 중첩 +0.0158
 

## 산출물

```
Yoonsoo/variants_a4_full/<물성>/ 중첩 변형 (22종 · 1,384,692행)
Yoonsoo/scores_a4_full/<물성>/ ChemBERTa 채점 (state_id 포함)
Yoonsoo/a4_decomposition_full/<물성>/ 정식 A²·B² 분해
Yoonsoo/a4_comparison_full/ 비교 3종 (셀 10 · 12 · 13)
```


이 노트북의 출력은 재실행 때 덮어쓴다. 항상 최신 1벌만 남는다.
읽기만 하는 곳: `Main/` · `Juhyeong/` · `Jiye/`.

접미사 없는 `scores_a4_chemberta` · `a4_decomposition` · `a4_comparison` 은
4물성 예비본이고 이 노트북은 손대지 않는다.

`compare_approx_formal.py` 는 폐기했다 — 근사 신호가 `variants_v2` 경로에
long 형식으로 있을 것을 전제하는데 A-3 산출물은 `scores_v2` 의 wide 형식이라,
돌리려면 출처 검사를 우회해야 한다. 같은 비교를 셀 10·12·13 이 한다.

## 읽는 법

- **상관이 높으면** 지금까지의 근사 결과가 유지된다. 근사를 쓴 근거가 생긴다
- **상관이 낮으면** 그 차이 자체가 발견이다. B 가 표기 효과에 오염돼 있었다는 뜻
- **(a)와 (b)가 갈리면** pH 창·B3 확장이 B 신호를 바꾼 것 — A-3 보고로 넘긴다